In [0]:
%run ../utils/utils

In [0]:
%run ../config/config

In [0]:
print(get_delta_path("gold", "gold_volumetria_tabelas", STORAGE_OPTIONS))

In [0]:
print("===== TABELAS DO CATALOGO =====")
spark.sql("SHOW TABLES IN squad1").show(truncate=False)

print("\n===== ARQUIVOS NO CONTAINER SQUAD1 =====")

file_system = service_client.get_file_system_client("squad1")

for item in file_system.get_paths(recursive=True):
    print(item.name)

In [0]:
print("===== TABELAS DO CATALOGO =====")
spark.sql("SHOW TABLES IN squad1").show(truncate=False)

print("\n===== ARQUIVOS NO CONTAINER SQUAD1 =====")

file_system = service_client.get_file_system_client("squad1")

for item in file_system.get_paths(recursive=True):
    print(item.name)

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Gold — Volumetria das Tabelas (Bronze / Silver / Quarentena / Falhas)
# MAGIC
# MAGIC Notebook de auditoria: percorre a lista de entidades do projeto, conta quantos
# MAGIC registros existem em cada camada física (Bronze, Silver e Silver/Quarentena) e
# MAGIC calcula o percentual real de falhas sobre o total da Bronze, consolidando tudo
# MAGIC em uma única tabela Gold (`gold_volumetria_tabelas`).
# MAGIC
# MAGIC ### Correções aplicadas nesta versão
# MAGIC 1. `total_registros_dq_logs` (contagem de LINHAS de log) foi renomeado para
# MAGIC    `total_eventos_dq_logs`, para deixar claro que é "quantas vezes uma regra
# MAGIC    falhou em alguma execução" — não é volume de dado reprovado.
# MAGIC 2. Adicionada `total_falhas_registros`, que é a SOMA de `qtd_registros_falhos`
# MAGIC    (o dado que realmente importa para calcular percentual de falha).
# MAGIC 3. Adicionada `percentual_falha`, calculada como
# MAGIC    `total_falhas_registros / total_registros_bronze_distintos * 100`, usando a
# MAGIC    contagem de IDs ÚNICOS na Bronze (não a contagem bruta de linhas, que pode
# MAGIC    inflar por duplicata física de ingestão).
# MAGIC 4. Aviso explícito no output: como `dq_monitoring_logs` é cumulativo
# MAGIC    (append-only) desde a criação da tabela, o percentual reflete o HISTÓRICO
# MAGIC    de todas as execuções já registradas, não só o estado atual da Bronze. Se
# MAGIC    a tabela de logs não foi resetada de forma consistente com Bronze/Silver
# MAGIC    (ex: durante testes e correções de regra), o percentual pode ficar
# MAGIC    artificialmente alto ou baixo.

# COMMAND ----------

# MAGIC %run ../utils/utils

# COMMAND ----------

# MAGIC %run ../config/config

# COMMAND ----------

import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType, TimestampType
from datetime import datetime, timezone

# COMMAND ----------

# MAGIC %md
# MAGIC ## Parâmetros

# COMMAND ----------

# ------------------------------------------------------------------
# Lista de entidades monitoradas. Ajuste conforme as tabelas do squad.
# ------------------------------------------------------------------
TABELAS_PROJETO = [
    "ecommerce_clientes",
    "ecommerce_pedidos",
    "ecommerce_enderecos",
    "ecommerce_itens_pedido",
    "ecommerce_rastreamento",
    "ecommerce_produtos",
    "ecommerce_categorias"
]

# Coluna de ID natural de cada tabela, usada para contar registros DISTINTOS na
# Bronze (protege o percentual contra duplicata física de ingestão — ver
# diagnóstico de reconciliação que fizemos para ecommerce_enderecos).
IDS_POR_TABELA = {
    "ecommerce_clientes": "id_cliente",
    "ecommerce_pedidos": "id_pedido",
    "ecommerce_enderecos": "id_endereco",
    "ecommerce_itens_pedido": "id_item_pedido",
    "ecommerce_rastreamento": "id_rastreamento",
    "ecommerce_produtos": "id_produto",
    "ecommerce_categorias": "id_categoria"
}

NOME_TABELA_GOLD = "gold_volumetria_tabelas"
TIMESTAMP_EXECUCAO = datetime.now(timezone.utc)

SCHEMA_GOLD_VOLUMETRIA = StructType([
    StructField("nome_tabela_origem", StringType(), True),
    StructField("total_registros_bronze", LongType(), True),
    StructField("total_registros_bronze_distintos", LongType(), True),
    StructField("total_registros_silver", LongType(), True),
    StructField("total_registros_quarentena", LongType(), True),
    StructField("total_eventos_dq_logs", LongType(), True),
    StructField("total_falhas_registros", LongType(), True),
    StructField("percentual_falha", DoubleType(), True),
    StructField("data_verificacao", TimestampType(), True),
])

print(f"===== INICIANDO VOLUMETRIA GOLD PARA {len(TABELAS_PROJETO)} TABELAS =====")
print("Tabela Gold de destino:", NOME_TABELA_GOLD)
print(
    "\nATENÇÃO: dq_monitoring_logs é cumulativo (append-only) desde sempre. "
    "O percentual de falha abaixo reflete TODO o histórico de execuções já "
    "gravado, não só o estado atual da Bronze. Se a tabela de logs não foi "
    "resetada junto com Bronze/Silver/Quarentena em algum momento (ex: durante "
    "testes), o percentual pode ficar bem diferente do que a Bronze atual sozinha "
    "sugeriria.\n"
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## Contagem por camada (Bronze / Silver / Quarentena / Falhas)

# COMMAND ----------

def contar_delta_seguro(camada: str, tabela: str) -> int:
    """Conta registros de um Delta; retorna 0 se a tabela não existir ou falhar a leitura."""
    try:
        return ler_delta(camada, tabela, STORAGE_OPTIONS).count()
    except Exception:
        return 0


def contar_distintos_bronze_seguro(tabela: str) -> int:
    """Conta IDs distintos na Bronze (protege contra duplicata física de ingestão)."""
    id_col = IDS_POR_TABELA.get(tabela)
    if id_col is None:
        return 0
    try:
        return ler_delta("bronze", tabela, STORAGE_OPTIONS).select(id_col).distinct().count()
    except Exception:
        return 0


# dq_monitoring_logs fica na RAIZ do container (camada=""), com o registro de
# erro de TODAS as tabelas. Lemos uma única vez fora do loop e filtramos por
# entidade, evitando reabrir o Delta a cada iteração.
if delta_existe("", "dq_monitoring_logs", STORAGE_OPTIONS):
    df_dq_logs = ler_delta("", "dq_monitoring_logs", STORAGE_OPTIONS)
    dq_logs_disponivel = True
    print(f"dq_monitoring_logs encontrada — total geral de linhas de log: {df_dq_logs.count()}")
else:
    df_dq_logs = None
    dq_logs_disponivel = False
    print("Aviso: tabela dq_monitoring_logs não encontrada na raiz do Data Lake.")

registros_volumetria = []

for tabela in TABELAS_PROJETO:
    qtd_bronze = contar_delta_seguro("bronze", tabela)
    qtd_bronze_distintos = contar_distintos_bronze_seguro(tabela)
    qtd_silver = contar_delta_seguro("silver", tabela)
    qtd_quarentena = contar_delta_seguro("silver/quarentena", tabela)

    if dq_logs_disponivel:
        df_logs_tabela = df_dq_logs.filter(F.col("tabela") == tabela)
        qtd_eventos_dq_logs = df_logs_tabela.count()
        # CORREÇÃO: soma real de registros reprovados, não contagem de linhas de log.
        total_falhas_registros = df_logs_tabela.agg(
            F.sum("qtd_registros_falhos")
        ).collect()[0][0] or 0
    else:
        qtd_eventos_dq_logs = 0
        total_falhas_registros = 0

    if qtd_bronze_distintos > 0:
        percentual_falha = round((total_falhas_registros / qtd_bronze_distintos) * 100, 2)
    else:
        percentual_falha = None

    registros_volumetria.append((
        tabela,
        qtd_bronze,
        qtd_bronze_distintos,
        qtd_silver,
        qtd_quarentena,
        qtd_eventos_dq_logs,
        int(total_falhas_registros),
        percentual_falha,
        TIMESTAMP_EXECUCAO,
    ))

    print(
        f"{tabela:30s} | bronze={qtd_bronze:>8} | bronze_distintos={qtd_bronze_distintos:>8} "
        f"| silver={qtd_silver:>8} | quarentena={qtd_quarentena:>8} "
        f"| eventos_log={qtd_eventos_dq_logs:>8} | falhas={total_falhas_registros:>8} "
        f"| perc_falha={percentual_falha}"
    )

df_gold_volumetria = spark.createDataFrame(registros_volumetria, schema=SCHEMA_GOLD_VOLUMETRIA)

print(f"\nTotal de linhas geradas: {df_gold_volumetria.count()}")
display(df_gold_volumetria)

# COMMAND ----------

# MAGIC %md
# MAGIC ## Gravação no Delta Lake (Gold)

# COMMAND ----------

sucesso_delta = gravar_delta(
    df=df_gold_volumetria,
    camada="gold",
    tabela=NOME_TABELA_GOLD,
    storage_opts=STORAGE_OPTIONS,
    mode="overwrite",
    particionar=False,
)

if sucesso_delta:
    print(f"[Sucesso] {NOME_TABELA_GOLD} gravada em Delta (pasta gold).")
else:
    print(f"[Erro] Falha ao gravar {NOME_TABELA_GOLD} em Delta.")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Réplica no SQL Server

# COMMAND ----------

if sucesso_delta:
    try:
        escrever_sqlserver_gold(
            df_spark=df_gold_volumetria,
            schema="squad1",
            tabela=NOME_TABELA_GOLD,
            modo="overwrite",
        )
        print(f"[Sucesso] {NOME_TABELA_GOLD} sincronizada no SQL Server.")
    except Exception as e:
        print(f"[Erro SQL Server] Falha ao enviar {NOME_TABELA_GOLD}: {e}")
else:
    print("[Aviso] Envio ao SQL Server ignorado: a gravação Delta falhou.")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Validação final

# COMMAND ----------

print("===== VALIDAÇÃO FINAL =====")

if delta_existe("gold", NOME_TABELA_GOLD, STORAGE_OPTIONS):
    df_validacao = ler_delta("gold", NOME_TABELA_GOLD, STORAGE_OPTIONS)
    print(f"Registros na tabela Gold {NOME_TABELA_GOLD}: {df_validacao.count()}")
    display(df_validacao)
else:
    print(f"Atenção: tabela {NOME_TABELA_GOLD} não encontrada na validação.")

print("===== PROCESSO CONCLUÍDO =====")